# Test Inference Workflow
Notebook per testare il workflow multi-agentico su `invoice-001.pdf`

In [ ]:
import os
import sys
import json

os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")

from dotenv import load_dotenv
load_dotenv()

print("OPEN_ROUTER_KEY configurata:", "Sì" if os.getenv("OPEN_ROUTER_KEY") else "No")

OPEN_ROUTER_KEY configurata: Sì


## Step 1: Estrai testo dal PDF con OCR

In [ ]:
from inferencev2 import extract_text_from_pdf

raw_text = extract_text_from_pdf("dataset/invoice-001.pdf")
print(raw_text[:1500])
print(f"\n... ({len(raw_text)} caratteri totali)")

2026-05-28 18:29:09.535 | INFO     | inferencev2:extract_text_from_pdf:48 - Estrazione OCR da: dataset/invoice-001.pdf


--- Page 1 ---
Invoice no: 51109338 et Om,

Date of issue: 04/13/2013 Ques

TECHSHOPY

Seller: Client:

Andrews, Kirby and Valdez Becker Ltd

58861 Gonzalez Prairie 8012 Stewart Summit Apt. 455
Lake Daniellefurt, IN 57228 North Douglas, AZ 95355

Tax Id: 945-82-2137 Tax Id: 942-80-0517

IBAN: GB75MCRL06841367619257

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
1, CLEARANCE! Fast Dell Desktop 3,00 each 209,00 627,00 10% 689,70
Computer PC DUAL CORE
WINDOWS 10 4/8/16GB RAM
2. HP T520 Thin Client Computer 5,00 each 37,75 188,75 10% 207,63
AMD GX-212JC 1.2GHz 4GB RAM
TESTED !!READ BELOW!!
3: gaming pc desktop computer 1,00 each 400,00 400,00 10% 440,00
4. 12-Core Gaming Computer 3,00 each 464,89 1 394,67 10% 1 534,14
Desktop PC Tower Affordable
GAMING PC 8GB AMD Vega RGB
5. Custom Build Dell Optiplex 9020 5,00 each 221,99 1 109,95 10% 1 220,95
MT i5-4570 3.20GHz Desktop
Computer PC
6. Dell Optiplex 990 MT Computer 4,00 each 269,95 1 079,80 10% 1 187,78
PC Quad Core 

## Step 2: Agente 1 - Analizza il documento
Deduce schema e struttura, estrae i dati.

In [ ]:
from inferencev2 import build_llm
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import re

llm = build_llm(temperature=0.2)

analyzer = create_agent(
    model=llm,
    tools=[],
    system_prompt="""Sei un analizzatore di documenti. Ricevi il testo OCR di un PDF generico.

Il tuo compito è produrre un JSON con DUE sezioni:

1. "schema": descrivi la struttura del documento. Per ogni campo indica:
   - "field": nome del campo
   - "type": uno tra "string", "number", "date", "list", "object"
   - "sensitive": true se il campo contiene dati personali/sensibili
   - "constraints": eventuali vincoli (es. "deve essere uguale alla somma di X")

2. "data": i dati estratti dal documento, strutturati secondo lo schema.

REGOLE FONDAMENTALI:
- NON fare assunzioni sul tipo di documento. Deduci la struttura dai dati presenti.
- Identifica automaticamente le relazioni tra campi.
- Marca come "sensitive: true" TUTTI i campi che contengono dati personali.
- Output SOLO JSON valido, senza markdown.
""",
)

result = analyzer.invoke({"messages": [HumanMessage(content=raw_text)]})
raw_output = result["messages"][-1].content

json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)
if json_match:
    raw_output = json_match.group(0)

parsed = json.loads(raw_output)
schema = parsed.get("schema", [])
source_data = parsed.get("data", {})

print("=== SCHEMA ===")
print(json.dumps(schema, indent=2))
print("\n=== DATI SORGENTE ===")
print(json.dumps(source_data, indent=2))

=== SCHEMA ===
{
  "invoice_details": {
    "type": "object",
    "fields": {
      "invoice_no": {
        "type": "string",
        "sensitive": false
      },
      "date_of_issue": {
        "type": "date",
        "sensitive": false
      }
    }
  },
  "seller": {
    "type": "object",
    "fields": {
      "name": {
        "type": "string",
        "sensitive": true
      },
      "address": {
        "type": "string",
        "sensitive": true
      },
      "tax_id": {
        "type": "string",
        "sensitive": true
      },
      "iban": {
        "type": "string",
        "sensitive": true
      }
    }
  },
  "client": {
    "type": "object",
    "fields": {
      "name": {
        "type": "string",
        "sensitive": true
      },
      "address": {
        "type": "string",
        "sensitive": true
      },
      "tax_id": {
        "type": "string",
        "sensitive": true
      }
    }
  },
  "items": {
    "type": "list",
    "fields": {
      "item_no": {
  

## Step 3: Agente 2 - Genera record sintetici

In [ ]:
N = 1  # numero di record da generare

llm_gen = build_llm(temperature=0.7)

generator = create_agent(
    model=llm_gen,
    tools=[],
    system_prompt=f"""Sei un generatore di dati sintetici. Devi generare ESATTAMENTE {N} record basati su schema e dati forniti.

REGOLE:
1. Genera ESATTAMENTE {N} record in un array JSON
2. Ogni record DEVE rispettare lo schema fornito (stessi campi, stessi tipi)
3. I campi marcati "sensitive": true DEVONO essere sostituiti con valori fittizi ma realistici
4. RISPETTA TUTTI i constraints dello schema (es. totali = somma dei dettagli)
5. Mantieni la coerenza interna: se un campo deriva da altri, ricalcolalo correttamente

Output: array JSON di {N} record. SOLO JSON, senza markdown.
""",
)

prompt = f"""SCHEMA del documento:
{json.dumps(schema, indent=2)}

DATI SORGENTE:
{json.dumps(source_data, indent=2)}

Genera {N} record sintetici."""

result = generator.invoke({"messages": [HumanMessage(content=prompt)]})
generated_raw = result["messages"][-1].content

json_match = re.search(r"\[.*\]", generated_raw, re.DOTALL)
if json_match:
    generated_raw = json_match.group(0)

generated = json.loads(generated_raw)
print(f"Generati {len(generated)} record sintetici")
print(json.dumps(generated[0], indent=2))

Generati 5 record sintetici
{
  "invoice_details": {
    "invoice_no": "INV-2023-001",
    "date_of_issue": "2023-01-15"
  },
  "seller": {
    "name": "TechSolutions Srl",
    "address": "Via Roma 12, Milano, MI 20121",
    "tax_id": "IT01234567890",
    "iban": "IT60X054240320000001234567"
  },
  "client": {
    "name": "Global Logistics SpA",
    "address": "Corso Italia 45, Torino, TO 10121",
    "tax_id": "IT09876543210"
  },
  "items": [
    {
      "item_no": 1,
      "description": "Dell Optiplex Desktop PC",
      "qty": 2,
      "unit_measure": "each",
      "net_price": 500.0,
      "net_worth": 1000.0,
      "vat_percentage": 22,
      "gross_worth": 1220.0
    },
    {
      "item_no": 2,
      "description": "HP Thin Client",
      "qty": 10,
      "unit_measure": "each",
      "net_price": 150.0,
      "net_worth": 1500.0,
      "vat_percentage": 22,
      "gross_worth": 1830.0
    }
  ],
  "summary": {
    "total_net_worth": 2500.0,
    "total_vat": 550.0,
    "total_gr

## Step 4: Agente 3 - Valida e correggi

In [ ]:
llm_val = build_llm(temperature=0.1)

validator = create_agent(
    model=llm_val,
    tools=[],
    system_prompt="""Sei un validatore di dati sintetici. Ricevi uno schema, dei constraints e dei record generati.

Per OGNI record:
1. Verifica che TUTTI i campi dello schema siano presenti
2. Verifica che i tipi siano corretti
3. Per ogni constraint nello schema, verifica che sia rispettato
4. Se trovi errori di calcolo, CORREGGI ricalcolando
5. Se un record è completamente invalido, rimuovilo

Output: array JSON con SOLO i record validi (corretti dove necessario).
NON aggiungere markdown, SOLO JSON.
""",
)

prompt = f"""SCHEMA e constraints:
{json.dumps(schema, indent=2)}

RECORD DA VALIDARE:
{json.dumps(generated, indent=2)}

Valida, correggi, e restituisci l'array JSON dei record validi."""

result = validator.invoke({"messages": [HumanMessage(content=prompt)]})
validated_raw = result["messages"][-1].content

json_match = re.search(r"\[.*\]", validated_raw, re.DOTALL)
if json_match:
    validated_raw = json_match.group(0)

validated = json.loads(validated_raw)
print(f"Record validati: {len(validated)}/{len(generated)}")
print(json.dumps(validated[0], indent=2))

Record validati: 5/5
{
  "invoice_details": {
    "invoice_no": "INV-2023-001",
    "date_of_issue": "2023-01-15"
  },
  "seller": {
    "name": "TechSolutions Srl",
    "address": "Via Roma 12, Milano, MI 20121",
    "tax_id": "IT01234567890",
    "iban": "IT60X054240320000001234567"
  },
  "client": {
    "name": "Global Logistics SpA",
    "address": "Corso Italia 45, Torino, TO 10121",
    "tax_id": "IT09876543210"
  },
  "items": [
    {
      "item_no": 1,
      "description": "Dell Optiplex Desktop PC",
      "qty": 2,
      "unit_measure": "each",
      "net_price": 500.0,
      "net_worth": 1000.0,
      "vat_percentage": 22,
      "gross_worth": 1220.0
    },
    {
      "item_no": 2,
      "description": "HP Thin Client",
      "qty": 10,
      "unit_measure": "each",
      "net_price": 150.0,
      "net_worth": 1500.0,
      "vat_percentage": 22,
      "gross_worth": 1830.0
    }
  ],
  "summary": {
    "total_net_worth": 2500.0,
    "total_vat": 550.0,
    "total_gross_wor

## Step 5: Confronto sorgente vs sintetico

In [ ]:
from IPython.display import display, HTML
import json

# 1. Normalizziamo lo 'schema' (se è una stringa testuale, la ritrasformiamo in lista)
if isinstance(schema, str):
    try:
        schema_list = json.loads(schema)
    except json.JSONDecodeError:
        schema_list = [] # Fallback di sicurezza
else:
    schema_list = schema

# (Opzionale ma consigliato) Facciamo lo stesso per 'source_data'
if isinstance(source_data, str):
    try:
        source_data_dict = json.loads(source_data)
    except json.JSONDecodeError:
        source_data_dict = {}
else:
    source_data_dict = source_data

# 2. Ora possiamo usare la lista sicura per estrarre i campi sensibili
sensitive_fields = [s["field"] for s in schema_list if isinstance(s, dict) and s.get("sensitive")]

# 3. Creazione della tabella HTML
html = "<table border='1' style='text-align: left;'><tr><th>Campo</th><th>Sorgente</th><th>Sintetico #1</th></tr>"

for field_def in schema_list:
    if not isinstance(field_def, dict):
        continue # Saltiamo eventuali elementi malformati
        
    field = field_def.get("field", "Sconosciuto")
    # Estraiamo i dati sicuri
    orig = str(source_data_dict.get(field, ""))[:50]
    synth = str(validated[0].get(field, ""))[:50] if validated else ""
    
    color = "red" if field_def.get("sensitive") else "green"
    html += f"<tr><td style='color:{color}; font-weight:bold;'>{field}</td><td>{orig}</td><td>{synth}</td></tr>"
    
html += "</table>"

display(HTML(html))


Campo,Sorgente,Sintetico #1



Campi sensibili individuati: []


## Bonus: Esegui l'intero workflow in un colpo solo

In [ ]:
from inferencev2 import build_graph
from langchain_core.messages import HumanMessage

app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="5"),
        HumanMessage(content="run_id:notebook_test"),
    ],
    "raw_text": raw_text,
    "document_schema": "",
    "source_data": "",
    "generated_data": "",
    "validated_data": "",
    "final_data": [],
    "errors": [],
})

print(f"Workflow completato: {len(result['final_data'])} record generati")
print(f"Salvati in: output/notebook_test/")